In [24]:
import os
import glob
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import matplotlib.pyplot as plt

# --- Configuration ---
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-4
NUM_CLASSES = 1   # 1 for binary mask, >1 for multi-class
AUTOTUNE = tf.data.AUTOTUNE

# Paths (edit these according to your dataset)
IMAGES_DIR = r"D:\ujjwal\programming\Deep Learning\dataset\images"
MASKS_DIR = r"D:\ujjwal\programming\Deep Learning\dataset\masks"


In [25]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def combined_loss(y_true, y_pred):
    if NUM_CLASSES == 1:
        ce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    else:
        ce = tf.keras.losses.CategoricalCrossentropy()(y_true, y_pred)
    return ce + dice_loss(y_true, y_pred)


In [26]:
def conv_block(x, filters, kernel_size=3, batchnorm=True):
    x = layers.Conv2D(filters, kernel_size, padding="same", kernel_initializer="he_normal")(x)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, kernel_size, padding="same", kernel_initializer="he_normal")(x)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2,2))(c)
    return c, p

def decoder_block(x, conv_features, filters):
    x = layers.Conv2DTranspose(filters, (2,2), strides=(2,2), padding="same")(x)
    x = layers.concatenate([x, conv_features])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(256, 256, 3), num_classes=1, base_filters=32):
    inputs = layers.Input(input_shape)
    
    # Encoder
    c1, p1 = encoder_block(inputs, base_filters)
    c2, p2 = encoder_block(p1, base_filters*2)
    c3, p3 = encoder_block(p2, base_filters*4)
    c4, p4 = encoder_block(p3, base_filters*8)
    
    # Bridge
    b1 = conv_block(p4, base_filters*16)
    
    # Decoder
    d1 = decoder_block(b1, c4, base_filters*8)
    d2 = decoder_block(d1, c3, base_filters*4)
    d3 = decoder_block(d2, c2, base_filters*2)
    d4 = decoder_block(d3, c1, base_filters)
    
    # Output
    if num_classes == 1:
        outputs = layers.Conv2D(1, (1,1), activation="sigmoid")(d4)
    else:
        outputs = layers.Conv2D(num_classes, (1,1), activation="softmax")(d4)
    
    model = models.Model(inputs, outputs, name="UNet")
    return model


In [27]:
def list_pairs(img_dir, mask_dir, extensions=("png", "jpg", "jpeg")):
    images = []
    for ext in extensions:
        images.extend(glob.glob(os.path.join(img_dir, f"*.{ext}")))
    images = sorted(images)
    pairs = []
    for img in images:
        mask = os.path.join(mask_dir, os.path.basename(img))
        if os.path.exists(mask):
            pairs.append((img, mask))
    return pairs

IMG_SIZE = (256, 256)

def load_image(img_path, mask_path):
    img = tf.io.read_file(img_path)
    mask = tf.io.read_file(mask_path)

    # If your dataset has JPGs, use decode_jpeg. For PNGs, use decode_png
    img = tf.image.decode_jpeg(img, channels=3)
    mask = tf.image.decode_png(mask, channels=1)

    # Ensure known shapes before resizing
    img.set_shape([None, None, 3])
    mask.set_shape([None, None, 1])

    img = tf.image.resize(img, IMG_SIZE)
    mask = tf.image.resize(mask, IMG_SIZE)

    # Normalize image and mask
    img = tf.cast(img, tf.float32) / 255.0
    mask = tf.cast(mask, tf.float32)

    # Convert mask to binary (optional: if it’s not already)
    mask = tf.where(mask > 0.5, 1.0, 0.0)

    return img, mask


def augment(img, mask):
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    return img, mask

def make_dataset(pairs, training=True):
    img_paths = [p[0] for p in pairs]
    mask_paths = [p[1] for p in pairs]

    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    ds = ds.map(lambda i, m: load_image(i, m), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: augment(x, y), num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(100)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


In [28]:
pairs = list_pairs(IMAGES_DIR, MASKS_DIR)
random.shuffle(pairs)

split = int(0.8 * len(pairs))
train_pairs = pairs[:split]
val_pairs = pairs[split:]

train_ds = make_dataset(train_pairs, training=True)
val_ds = make_dataset(val_pairs, training=False)

print(f"Total pairs: {len(pairs)} | Train: {len(train_pairs)} | Val: {len(val_pairs)}")


Total pairs: 3064 | Train: 2451 | Val: 613


In [29]:
model = build_unet(input_shape=(256,256,3), num_classes=NUM_CLASSES)
model.compile(optimizer=tf.keras.optimizers.Adam(LR),
              loss=combined_loss,
              metrics=[dice_coef, iou_coef, "accuracy"])

model.summary()


Model: "UNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 256, 256,  │        896 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_31[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_30       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 256, 256,  │      9,248 │ activation_30[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_32[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_31       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 128, 128,  │          0 │ activation_31[0]… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 128, 128,  │     18,496 │ max_pooling2d_8[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_33[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_32       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 128, 128,  │     36,928 │ activation_32[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_34[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_33       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 64, 64,    │          0 │ activation_33[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_9[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_35[0][0] 

 Total params: 7,771,873 (29.65 MB)

 Trainable params: 7,765,985 (29.62 MB)

 Non-trainable params: 5,888 (23.00 KB)

In [30]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

callbacks = [
    ModelCheckpoint("unet_best_150.h5", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Epoch 1/40
  9/307 ━━━━━━━━━━━━━━━━━━━━ 12:55 3s/step - accuracy: 0.8901 - dice_coef: 0.0398 - iou_coef: 0.0203 - loss: 1.3518

KeyboardInterrupt: 